# Step 4 — 按 prompt 切分 train / validation / test

同一题的所有 rollout 必须属于同一个 split，避免 trajectory-level leakage。

默认使用 80% / 10% / 10%，并验证三个集合的 prompt 完全不重叠。

In [ ]:
# @title Step 04.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow','tqdm']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    progress_frame,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

CONFIG = load_config(REPO)
print("Experiment:", CONFIG["experiment_name"])


In [ ]:
# @title Step 04.2 — 查看持久化进度
runtime_progress = progress_frame(REPO)
print("持久化进度快照：")
display(runtime_progress if not runtime_progress.empty else pd.DataFrame([{"状态": "尚无进度记录"}]))

In [ ]:
# @title Step 04.3 — 按 prompt 创建 train / validation / test
import numpy as np
import pandas as pd
from IPython.display import display

source_path = REPO / CONFIG["paths"]["full"]
df = pd.read_parquet(source_path)
prompt_ids = np.array(sorted(df["prompt_idx"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(prompt_ids)

n_prompts = len(prompt_ids)
train_end = int(n_prompts * CONFIG["train_fraction"])
validation_end = train_end + int(n_prompts * CONFIG["validation_fraction"])
assignments = {
    "train": set(prompt_ids[:train_end].tolist()),
    "validation": set(prompt_ids[train_end:validation_end].tolist()),
    "test": set(prompt_ids[validation_end:].tolist()),
}

split_frames = {}
for name, ids in assignments.items():
    split_frame = df[df["prompt_idx"].isin(ids)].copy().reset_index(drop=True)
    split_frame["data_split"] = name
    path = REPO / CONFIG["paths"][name]
    path.parent.mkdir(parents=True, exist_ok=True)
    split_frame.to_parquet(path, index=False)
    split_frames[name] = split_frame
    print(name, path, len(split_frame))

In [ ]:
# @title Step 04.4 — 检查分布稳定性与数据泄漏
import matplotlib.pyplot as plt

summary = pd.DataFrame([
    {
        "split": name,
        "prompts": frame["prompt_idx"].nunique(),
        "rollouts": len(frame),
        "accuracy": frame["correct"].mean(),
        "finished_rate": frame["finished"].mean(),
        "median_length": frame["length"].median(),
    }
    for name, frame in split_frames.items()
])
display(summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
summary.set_index("split")[["prompts", "rollouts"]].plot.bar(ax=axes[0])
axes[0].set_title("Split sizes")
summary.set_index("split")["accuracy"].plot.bar(ax=axes[1], color="#49beaa", ylim=(0, 1))
axes[1].set_title("Correctness stability")
for name, frame in split_frames.items():
    axes[2].hist(frame["length"], bins=30, histtype="step", density=True, label=name)
axes[2].set(title="Length distributions", xlabel="tokens")
axes[2].legend()
plt.tight_layout()
plt.show()

overlaps = {
    "train∩validation": len(assignments["train"] & assignments["validation"]),
    "train∩test": len(assignments["train"] & assignments["test"]),
    "validation∩test": len(assignments["validation"] & assignments["test"]),
}
checks = [
    gate("无 prompt 泄漏", sum(overlaps.values()) == 0, str(overlaps)),
    gate("所有数据均已分配", sum(len(frame) for frame in split_frames.values()) == len(df), f"{sum(len(frame) for frame in split_frames.values())}/{len(df)}"),
    gate("每个 split 正负标签都有", all(frame["correct"].nunique() == 2 for frame in split_frames.values()), str({name: frame['correct'].value_counts().to_dict() for name, frame in split_frames.items()}), kind="scientific"),
    gate("Split accuracy 漂移 ≤10pp", summary["accuracy"].max() - summary["accuracy"].min() <= .10, f"range={summary['accuracy'].max() - summary['accuracy'].min():.1%}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "04_prompt_split", checks, {"summary": summary.to_dict(orient="records"), "overlaps": overlaps})